# Delhi air quality: explore the data

Delhi is our single city. Each row represents one calendar day. Unit I covers pandas, and Unit IV covers cleaning and exploration.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.data_preparation import load_city_data, build_features, build_targets
daily = load_city_data()


In [2]:
print(daily.shape)
print(daily.head())
print(daily.dtypes)
print(daily.describe())

(2009, 15)
             City   PM2.5    PM10     NO    NO2     NOx     NH3     CO   SO2  \
Date                                                                           
2015-01-01  Delhi  313.22  607.98  69.16  36.39  110.59   33.85  15.20  9.25   
2015-01-02  Delhi  186.18  269.55  62.09  32.87   88.14   31.83   9.54  6.65   
2015-01-03  Delhi   87.18  131.90  25.73  30.31   47.95   69.55  10.61  2.65   
2015-01-04  Delhi  151.84  241.84  25.01  36.91   48.62  130.36  11.54  4.63   
2015-01-05  Delhi  146.60  219.13  14.01  34.92   38.25  122.88   9.20  3.33   

               O3  Benzene  Toluene  Xylene    AQI AQI_Bucket  
Date                                                           
2015-01-01  41.68    14.36    24.86    9.84  472.0     Severe  
2015-01-02  29.97    10.55    20.09    4.29  454.0     Severe  
2015-01-03  19.71     3.91    10.23    1.99  143.0   Moderate  
2015-01-04  25.36     4.26     9.71    3.34  319.0  Very Poor  
2015-01-05  23.20     2.80     6.21    2.96 

In [3]:
print("Missing percent")
print(daily.isnull().sum() / len(daily) * 100)
print("Missing percent by year")
print(daily.isnull().groupby(daily.index.year).mean() * 100)

Missing percent
City           0.000000
PM2.5          0.099552
PM10           3.832753
NO             0.099552
NO2            0.099552
NOx            0.000000
NH3            0.447984
CO             0.000000
SO2            5.475361
O3             4.181185
Benzene        0.000000
Toluene        0.000000
Xylene        38.875062
AQI            0.497760
AQI_Bucket     0.497760
dtype: float64
Missing percent by year
      City     PM2.5       PM10        NO       NO2  NOx      NH3   CO  \
Date                                                                     
2015   0.0  0.000000   1.917808  0.000000  0.000000  0.0  0.00000  0.0   
2016   0.0  0.000000   2.459016  0.000000  0.000000  0.0  1.36612  0.0   
2017   0.0  0.547945  16.712329  0.547945  0.547945  0.0  1.09589  0.0   
2018   0.0  0.000000   0.000000  0.000000  0.000000  0.0  0.00000  0.0   
2019   0.0  0.000000   0.000000  0.000000  0.000000  0.0  0.00000  0.0   
2020   0.0  0.000000   0.000000  0.000000  0.000000  0.0  0.00000  

In [4]:
print("Duplicate dates:", daily.index.duplicated().sum())
print("First and last dates:", daily.index.min(), daily.index.max())
print(daily["AQI_Bucket"].value_counts())

Duplicate dates: 0
First and last dates: 2015-01-01 00:00:00 2020-07-01 00:00:00
AQI_Bucket
Poor            542
Very Poor       520
Moderate        519
Severe          239
Satisfactory    158
Good             21
Name: count, dtype: int64


In [5]:
plt.figure(figsize=(10, 4))
plt.plot(daily.index, daily["PM2.5"])
plt.xlabel("Date")
plt.ylabel("PM2.5 (ug/m3)")
plt.title("Delhi PM2.5 over time")
plt.tight_layout()
plt.show()

C:\Users\renun\AppData\Local\Packages\openai.codex_2p2nqsd0c76g0\AC\#!sandbox.{d3280143-5b1b-48b9-8cc8-6a96ba6dadd5}\Temp\ipykernel_23368\123723089.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The graph shows annual peaks and sharp daily changes. Early 2020 is lower than the preceding winter peak. The plot does not establish a causal lockdown effect. Xylene is missing on 38.88% of days and is excluded. We use five pollutants, carry inputs forward at most two days, and fit any remaining median imputation on training data only. We retain observed targets and pollution spikes.